In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
data = pd.read_csv('input_data.csv', sep = ';')
data.head()

### Оптимизация памяти и чтение данных

In [ ]:
shape = data.shape
print(f'Dataset has {shape[0]} objects and {shape[1]} features')

In [ ]:
data.info()

In [ ]:
for col in ['kitchen_area', 'area', 'postal_code', 'street_id', 'house_id']:
    clean = data[col].dropna()
    print(f"{col}: {'✅ int' if (clean == clean.astype(int)).all() else '❌ float'}")

In [ ]:
dtypes_optimized = {
    'price': 'int64',     
    'level': 'int8',      
    'levels': 'int8',      
    'rooms': 'int8',
    'area': 'float32',
    'kitchen_area': 'float32',
    'geo_lat': 'float32',
    'geo_lon': 'float32',
    'building_type': 'int8',
    'object_type': 'int8',
    'postal_code': 'int32',
    'street_id': 'int32',
    'id_region': 'uint8',
    'house_id': 'int32'     
}
limits = {}

for dtype in set(dtypes_optimized.values()):
    if 'int' in dtype:
        limits[dtype] = np.iinfo(dtype).max
    elif 'float' in dtype:
        limits[dtype] = np.finfo(dtype).max

def get_min_limit(dtype):
    if dtype and 'int' in dtype:
        return np.iinfo(dtype).min
    elif dtype and 'float' in dtype:
        return np.finfo(dtype).min
    return '?'

df_limits = pd.DataFrame({
    'column' : data.max().index,
    'max_value' : data.max().values,
    'min_value' : data.min().values,
    'dtype' : [dtypes_optimized.get(col, '?') for col in data.max().index],
    'max_limit' : [limits.get(dtypes_optimized.get(col), '?') for col in data.max().index],
    'min_limit' : [get_min_limit(dtypes_optimized.get(col)) for col in data.max().index],
    'check_max' : [data.max()[col] <= limits.get(dtypes_optimized.get(col), 1e308) 
                  if dtypes_optimized.get(col) in limits else '❌' 
                  for col in data.max().index],
    'check_min' : [data.min()[col] >= get_min_limit(dtypes_optimized.get(col))
                  if dtypes_optimized.get(col) in limits else '❌'
                  for col in data.max().index]}).drop([0])
df_limits

In [ ]:
dtypes_optimized['postal_code'] = 'Int32'
dtypes_optimized['street_id'] = 'Int32'
dtypes_optimized['house_id'] = 'Int32'     

In [ ]:
df = pd.read_csv('input_data.csv', sep = ';', parse_dates = ['date'], dtype = dtypes_optimized)
df.head()

In [ ]:
df.info()

**ВАЖНО:** Сократили память в два раза без потерь по качеству.

### Предварительная статистика и проверка на целостность/искажения

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
df.describe()

Имеем большое количество пропусков в столбцах с __street_id__, __postal_code__ и __house_id__. Так же следует заметить, что у нас присутствуют отрицателнные значения в колонках __rooms__ и __kitchen__. Автор датасета дает комментарий касаемо -1 в комнатах, говоря о том, что этот тип данных соответствует студии, предлагается в дальнейшем присвоить ему 0. __kitchen__ требует внимательного рассмотрения причин, является ли эта строчка явным выбросом или есть какие-то иные причины. Наличие квартир с нулевой стоимостью одназначно говорит о какой-то ошибке, почти с полной уверенностью можно считать это некой аномалией в данных.

In [ ]:
df.isna().any()

Пока что поработаем с другими столбцами, вернемся к заполнению пропусков позднее.

In [ ]:
df[df['kitchen_area'] < 0]

Было бы логично считать, что в соответствие маркеру студии ставим отрицательное значение кухни, одна это вовсе не так и значение меньше 0 присутствует у абсолютно разных по характеристикам квартир. Имеем право считать, что такой сигнал в данных может быть маяком для пропусков, либо неудачно собранных данных, поэтому заменим все проблемные значения на NaN.

In [ ]:
df['kitchen_area'] = df['kitchen_area'].where(df['kitchen_area'] > 0, np.nan)
df[df['kitchen_area'] < 0].shape[0]

Поработаем с ценовой политикой данных, можно обратить внимание, что встречаются нулевые значения. Посмотрим в целом на распределение данных.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize = (15, 5))

pos_prices = df[df['price'] > 0]['price']
prices = pos_prices.values
prices_filtered = prices[prices <= np.percentile(prices, 99)]

axes[0].hist(prices_filtered, bins = 100, edgecolor = 'black', alpha = 0.7, color = 'steelblue')
axes[0].set_xlabel('Цена')
axes[0].set_ylabel('Частота')
axes[0].set_xlim(0)
axes[0].set_title('Распределение цен до 99-го перцентиля')
axes[0].axvline(x = 0, color = 'red', linestyle = '--', linewidth = 1, label = 'price = 0')
axes[0].legend()

log_prices = np.log10(pos_prices)
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99, 99.5, 99.9]
values = np.percentile(log_prices, percentiles)

axes[1].plot(percentiles, values, 'o-', linewidth = 2, markersize = 6, color = 'steelblue')
axes[1].fill_between(percentiles, values, alpha = 0.2)
axes[1].set_xscale('log')
axes[1].set_xlabel('Процентиль (лог. шкала)')
axes[1].set_ylabel('log10(Цена)')
axes[1].set_title('Процентили распределения')
axes[1].grid(True, alpha = 0.3)

df_pos = df[df['price'] > 0]
axes[2].hist(np.log10(df_pos['price']), bins = 100, edgecolor = 'black', alpha = 0.7, color = 'green')
axes[2].set_xlabel('log10(Цена)')
axes[2].set_ylabel('Частота')
axes[2].set_xlim(0)
axes[2].set_title('Распределение цен')
axes[2].axvline(x = np.log10(100000), color = 'red', linestyle = '--', alpha = 0.5, label = '100k')
axes[2].axvline(x = np.log10(1000000), color = 'orange', linestyle = '--', alpha = 0.5, label = '1M')
axes[2].axvline(x = np.log10(10000000), color = 'green', linestyle = '--', alpha = 0.5, label = '10M')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (12, 10))

plt.hist(prices_filtered, bins = 200, density = True, alpha = 0.5, color = 'steelblue', edgecolor = 'black')

shape, loc, scale = stats.lognorm.fit(prices_filtered, floc = 0)
x = np.linspace(0.01, max(prices_filtered), 1000)
pdf_lognorm = stats.lognorm.pdf(x, shape, loc, scale)
plt.plot(x, pdf_lognorm, 'r-', linewidth = 2, label = f'Логнормальное (shape = {shape:.2f})')
plt.xlabel('Цена')
plt.ylabel('Плотность')
plt.title('Логнормальное распределение')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
ksstat, kspvalue = stats.kstest(prices_filtered, 'lognorm', args = (shape, loc, scale))
print(f"Статистика Kолмогорова-Смирнова: {ksstat:.6f}")
print(f"p-value: {kspvalue:.10f}")

Как можем увидеть, распределение цены с очень высокой точностью описывается логнормальным распределением. Это довольно полезное свойство, которое следует иметь ввиду при дальнейшем рассмотрении и анализе на выбросы/искажения в данных. Помимо этого, как один вариантов примитивной статистической модели будем использовать логнормальное распределение. Собственно говор boxplot говорит нам о том же самом, что SADASDASD

In [ ]:
low_price df[df['price'] <= 100000] 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

top_regions = zero_price['id_region'].value_counts().head(15)
axes[0].barh(range(len(top_regions)), top_regions.values, color = 'salmon')
axes[0].set_yticks(range(len(top_regions)))
axes[0].set_yticklabels(top_regions.index)
axes[0].set_xlabel('Количество квартир с price = 0')
axes[0].set_title('Топ-15 регионов по количеству нулевых цен')
axes[0].invert_yaxis()

for i, v in enumerate(top_regions.values):
    axes[0].text(v + 100, i, f'{v:,}', va = 'center')

zero_by_month = zero_price.groupby(zero_price['date'].dt.to_period('M')).size()
normal_by_month = normal_price.groupby(normal_price['date'].dt.to_period('M')).size()

axes[1].plot(range(len(zero_by_month)), zero_by_month.values, marker='o', 
             label = 'price = 0', color = 'red', linewidth = 2, markersize = 4)
axes[1].plot(range(len(normal_by_month)), normal_by_month.values / 1000, 
             marker = 's', label = 'price>0 (тыс.)', color = 'blue', alpha = 0.7, linewidth = 2, markersize = 4)
axes[1].set_xlabel('Месяц (янв 2021 - дек 2021)')
axes[1].set_ylabel('Количество объявлений')
axes[1].set_title('Динамика объявлений по месяцам')
axes[1].legend()
axes[1].grid(True, alpha = 0.3)

months = [str(p)[:7] for p in zero_by_month.index]
axes[1].set_xticks(range(0, len(months), 2))
axes[1].set_xticklabels([months[i] for i in range(0, len(months), 2)], rotation = 45)

plt.tight_layout()
plt.show()